# VAR（向量自回歸，1980）

- **核心原理**：多條序列互相回歸
- **變體**：VECM（向量誤差修正模型，處理共整合）、結構 VAR（SVAR）、貝氏 VAR
- **推薦資料集**：美國總經資料（`statsmodels.datasets.macrodata`：1959Q1–2009Q3 每季，203 季）—— 本章用實質 GDP、實質消費、實質投資三條序列的**季成長率**
- **實作**：✋ 手刻（每條方程式 OLS）；🧰 statsmodels `VAR` 對照
- **文獻**：Sims (1980), *Macroeconomics and Reality*；Lütkepohl (2005), *New Introduction to Multiple Time Series Analysis*

本章依照 README 的「每章標準流程」：EDA → Baseline → 手寫模型 → 套件實作 → 調參 → 測試集評估 → 診斷 → 本章小結。

和第 2 章（ARIMA 的 AR 部分）的差別：

| | 第 2 章 AR($p$) | 本章 VAR($p$) |
| --- | --- | --- |
| 序列 | 一條 $z_t$（純量） | $k$ 條一起：$\mathbf y_t = (y_{1t}, \dots, y_{kt})$（向量） |
| 模型 | $z_t = c + \sum_j \phi_j z_{t-j} + \varepsilon_t$ | $\mathbf y_t = \mathbf c + \sum_j A_j\,\mathbf y_{t-j} + \boldsymbol\varepsilon_t$，$A_j$ 是 $k \times k$ 矩陣 |
| 新的問題 | — | GDP 的過去能幫忙預測投資嗎？（**Granger 因果**）一個「消費衝擊」會怎麼傳到 GDP？（**脈衝響應**） |
| 估計 | OLS（lag 矩陣） | **每條方程式各自 OLS**（右邊的變數都一樣 → 一次 `lstsq` 解完 $k$ 條） |

一句話：**VAR 把 $k$ 條序列的「過去」全部當成特徵，對每一條序列的「現在」各做一個線性回歸 —— 所以每條序列都可以用其他序列的過去來預測。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A–3D、4、5 節開頭另有各自的符號表或名詞表。

**資料**：`load_macrodata()` 回傳 statsmodels 的 macrodata（203 季、14 欄）。本章取 `realgdp`、`realcons`、`realinv`（實質 GDP、消費、投資，十億美元，2005 年幣值，季調後），轉成**季成長率（%）** $100\,\Delta\log$ → 202 季（1959Q2–2009Q3）。最後 12 季（2006Q4–2009Q3，含 2008 金融危機）當測試集。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 本章的值 |
| --- | --- | --- |
| $k$ | 序列的條數 | 3 |
| $\mathbf y_t$ | 第 $t$ 季的三個成長率（列向量） | $(k,)$ |
| $p$ | 落後期數（lag） | 1～8 |
| $\mathbf c$ | 常數向量 | $(k,)$ |
| $A_j$ | 第 $j$ 個 lag 的係數矩陣：$A_j[i, l]$ = 「序列 $l$ 的 $t - j$ 期」對「序列 $i$ 的 $t$ 期」的影響 | $(k, k)$ |
| $B$ | 把 $\mathbf c$、$A_1, \dots, A_p$ 疊起來的係數矩陣（OLS 的解） | $(1 + kp, k)$ |
| $\Sigma_u$ | 殘差的共變異數矩陣 | $(k, k)$ |
| $\Psi_h$ | 脈衝響應：第 0 期序列 $l$ 多 1 單位的衝擊，$h$ 期後序列 $i$ 變多少 | $(k, k)$ |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第 2 章 | ARIMA（`06-time-series/02-arima`） | AR($p$) = 對 lag 矩陣做 OLS；遞迴預測（預測值回填當下一步的輸入）；**比較不同 $p$ 的 AIC 時，要用同樣的資料點**（OLS 丟掉前 $p$ 個點）；平穩條件：特徵根在單位圓外。 |
| 第 1、2 章 | 指數平滑、ARIMA | MASE、滾動起點 CV、one-standard-error rule（fold 太少時標準誤很大）。 |
| 第一部分第 2 章 | 線性回歸 | 多元 OLS：$\hat\beta = (X^\top X)^{-1}X^\top y$；**多個 $y$ 共用同一個 $X$** 時，`lstsq(X, Y)` 一次解出每一欄。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 成長率與 log 差分**：$100(\log Y_t - \log Y_{t-1}) \approx 100\frac{Y_t - Y_{t-1}}{Y_{t-1}}$（%）。例：$Y$ 從 100 到 101 → 0.995%。成長率通常是平穩的，水準（GDP 本身）有趨勢、不平穩（第 2 章）。

**2. 矩陣乘向量**：$A\mathbf y$ 的第 $i$ 個元素 $= \sum_l A[i, l]\,y_l$。例：$A = \begin{pmatrix}0.5 & 0\\0.1 & 0.5\end{pmatrix}$、$\mathbf y = (2, 1)$ → $A\mathbf y = (1.0, 0.7)$。

**3. 多個目標的 OLS**：$Y$ $(n, k)$ 的每一欄是一個目標、共用同一個 $X$ → $B = \arg\min\lVert Y - XB\rVert^2$ 的每一欄就是各自的 OLS 解。`np.linalg.lstsq(X, Y)` 直接回傳 $(d, k)$ 的 $B$。

**4. 行列式與「廣義變異數」**：$\det\Sigma$ = 多維的變異數大小。VAR 的 AIC $= \log\det\hat\Sigma_u + \frac{2\,(\text{參數個數})}{T}$，$\hat\Sigma_u = \frac{1}{T}U^\top U$（$U$ 是殘差矩陣、$T$ 是用來估計的期數）。

**5. 矩陣的冪次**：VAR(1) 的 $h$ 期後的效果 $= A^h$；$A^h \to 0$（所有特徵值的絕對值 < 1）⇔ 平穩。

**6. F 檢定**（Granger 因果）：比較「有某些變數」與「沒有某些變數」兩個回歸的殘差平方和；差得夠多 → 那些變數有預測力。

**7. Cholesky 分解**：正定矩陣 $\Sigma = PP^\top$，$P$ 是下三角。用在「正交化」的脈衝響應：把相關的衝擊拆成彼此獨立的衝擊（結果跟變數的**排列順序**有關）。

#### 程式

**8. `np.column_stack`、`np.linalg.lstsq`**（第 2 章）。**9. `B[a:b].T`**：切出一塊再轉置。**10. `pd.PeriodIndex([...], freq="Q")`**：季資料的索引（`"1959Q1"`）。

---
## 1. 資料與探索性資料分析（EDA）

In [ ]:
import itertools
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from learning_map.data import load_macrodata

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
macro = load_macrodata()
macro.index = pd.PeriodIndex([f"{int(a)}Q{int(b)}" for a, b in zip(macro["year"], macro["quarter"])], freq="Q")
COLS = ["realgdp", "realcons", "realinv"]
levels = macro[COLS]
growth = (100 * np.log(levels).diff()).dropna()
K = len(COLS)
TEST_Q = 12
g_train, g_test = growth.iloc[:-TEST_Q], growth.iloc[-TEST_Q:]
print("季成長率：", growth.shape, growth.index[0], "～", growth.index[-1], "；train", len(g_train), "季；test", g_test.index[0], "～", g_test.index[-1])

fig, axes = plt.subplots(2, 1, figsize=(14, 6.5))
(levels / levels.iloc[0]).plot(ax=axes[0], logy=True)
axes[0].set_title("levels (1959Q1 = 1, log scale)")
growth.plot(ax=axes[1], lw=0.8)
axes[1].axvspan(g_test.index[0].to_timestamp(), g_test.index[-1].to_timestamp(), color="gray", alpha=0.15)
axes[1].set_title("quarterly growth (%), test period shaded")
plt.tight_layout()
plt.show()
display(g_train.describe().round(3).T)

In [ ]:
# 同期與落後的相關（cross-correlation）
# 觀察：
#   - 三條成長率「同一季」的相關係數各是多少？哪兩條最相關？
#   - 「投資的上一季」跟「GDP 的這一季」相關嗎？反過來呢？—— 這是 VAR 能利用的資訊
def lagged_corr(a: pd.Series, b: pd.Series, lag: int) -> float:
    """corr(a_t, b_{t−lag})。"""
    return float(np.corrcoef(a.iloc[lag:].to_numpy(), b.iloc[: len(b) - lag].to_numpy())[0, 1])


display(g_train.corr().round(3))
rows = []
for target, source in itertools.product(COLS, COLS):
    rows.append({"target (t)": target, "source (t−lag)": source, **{f"lag {L}": lagged_corr(g_train[target], g_train[source], L) for L in [1, 2, 4]}})
display(pd.DataFrame(rows).round(3).pivot(index="target (t)", columns="source (t−lag)", values="lag 1"))

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 水準 vs 成長率：水準的圖（log 尺度）大致是直線 —— 代表什麼？為什麼本章用成長率，不直接用水準？（第 2 章：平穩）
# (2) 三條成長率的平均、標準差：哪一條的波動最大？大幾倍？（提示：投資是景氣循環裡最劇烈的部分）
# (3) 同期相關：GDP 與消費、GDP 與投資各多少？（GDP ≈ 消費 + 投資 + 政府支出 + 淨出口）
# (4) lag 1 的相關表：哪一個「source → target」最強？是「自己的過去」還是「別人的過去」？
#     如果「別人的過去」也有預測力，單變數的 AR 就會漏掉這個資訊 —— 這就是 VAR 的動機。
# 答：

---
## 2. Baseline

| Baseline | 一步預測 $\hat y_{i,t\mid t-1}$ |
| --- | --- |
| 歷史平均 | train 的平均成長率 |
| naive | $y_{i,t-1}$（上一季的成長率） |
| **單變數 AR($p$)** | 每條序列各自用第 2 章的 AR（只看自己的過去）—— VAR 要贏它，才代表「別人的過去」有用 |

**評估方式**：在 test 的 12 季做**滾動的一步預測** —— 每一季都用「到上一季為止的真實資料」預測這一季（參數固定用 train 估的）。這是總經預測（nowcasting）最常見的用法。指標：每條序列的 RMSE、MAE、MASE（`season=1`：跟 naive 比），以及三條的平均 MASE。

（2008 年金融危機在 test 期間 —— 任何只看過去的線性模型都很難預測這種衝擊；重點是**相對**於 baseline 的表現。）

In [ ]:
from learning_map.metrics import forecast_metrics

Y_all = growth.to_numpy()
N_TR = len(g_train)


def score_one_step(pred: np.ndarray, name: str) -> dict:
    """pred (TEST_Q, K)：test 每一季的一步預測。回傳每條的 rmse、mase 與平均 mase。"""
    out = {}
    for i, c in enumerate(COLS):
        m = forecast_metrics(g_test[c], pred[:, i], g_train[c], season=1)
        out[f"{c} rmse"], out[f"{c} mase"] = m["rmse"], m["mase"]
    out["mean mase"] = round(float(np.mean([out[f"{c} mase"] for c in COLS])), 4)
    return out


def univariate_ar_one_step(p: int) -> np.ndarray:
    """每條序列各自的 AR(p)（OLS、有常數）→ test 的一步預測 (TEST_Q, K)。"""
    pred = np.zeros((TEST_Q, K))
    for i in range(K):
        z = Y_all[:N_TR, i]
        X = np.column_stack([np.ones(N_TR - p)] + [z[p - j : N_TR - j] for j in range(1, p + 1)])
        beta = np.linalg.lstsq(X, z[p:], rcond=None)[0]
        for s in range(TEST_Q):
            t = N_TR + s
            pred[s, i] = beta[0] + sum(beta[j] * Y_all[t - j, i] for j in range(1, p + 1))
    return pred


test_table = pd.DataFrame({
    "historical mean": score_one_step(np.tile(g_train.mean().to_numpy(), (TEST_Q, 1)), "mean"),
    "naive": score_one_step(Y_all[N_TR - 1 : N_TR - 1 + TEST_Q], "naive"),
    "univariate AR(1)": score_one_step(univariate_ar_one_step(1), "ar1"),
    "univariate AR(2)": score_one_step(univariate_ar_one_step(2), "ar2"),
}).T
display(test_table)

In [ ]:
# TODO 2（問答）Baseline
#
# (1) 哪一個 baseline 的平均 MASE 最低？naive 的 MASE 為什麼是 1（或很接近 1）？（提示：MASE 的分母）
# (2) 歷史平均 vs naive：哪條序列 naive 比較好、哪條平均比較好？跟 lag 1 的自我相關（EDA）有什麼關係？
# (3) 先寫下預測：VAR 會比單變數 AR 好多少？三條序列中，哪一條最可能受惠？（EDA 的 lag 1 相關表）
# 答：

---
## 3. 手寫模型：VAR($p$)

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | 設計矩陣：常數 + $p$ 個 lag 的 $k$ 條序列 | 3 |
| 3B | OLS（每條方程式一起解）、殘差共變異、AIC | 4 |
| 3C | 遞迴預測 | 5 |
| 3D | 脈衝響應 $\Psi_h$ | 6 |
| 3E | 對照 statsmodels；實驗：Granger 因果、正交化脈衝響應、選 $p$ | 7、8 |

**程式結構**：`VARScratch(p)` —— `fit(Y)` 估計 `B_`（$(1 + kp, k)$）、`intercept_`（$\mathbf c$）、`coefs_`（$p$ 個 $A_j$ 疊成 $(p, k, k)$）、`sigma_u_`、`aic_`；`forecast(y_last, h)`、`one_step(Y_all, start)`、`irf(h)`。

### 3A. 設計矩陣

$$\underbrace{\begin{pmatrix}\mathbf y_{p+1}^\top\\ \vdots \\ \mathbf y_n^\top\end{pmatrix}}_{\text{target } (n-p,\ k)} = \underbrace{\begin{pmatrix} 1 & \mathbf y_p^\top & \mathbf y_{p-1}^\top & \cdots & \mathbf y_1^\top\\ \vdots \\ 1 & \mathbf y_{n-1}^\top & \mathbf y_{n-2}^\top & \cdots & \mathbf y_{n-p}^\top\end{pmatrix}}_{X\ (n-p,\ 1 + kp)} \underbrace{\begin{pmatrix}\mathbf c^\top\\ A_1^\top\\ \vdots\\ A_p^\top\end{pmatrix}}_{B\ (1 + kp,\ k)} + U$$

（跟 statsmodels 的 `params` 同樣的排列：先常數，再 lag 1 的 $k$ 條、lag 2 的 $k$ 條…）

#### 符號表（3A–3D）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $Y$ | 序列（0 起算），每一列一季 | $(n, k)$ | `Y` |
| $X$ | 設計矩陣 | $(n - p, 1 + kp)$ | `X` |
| target | $Y[p:]$ | $(n - p, k)$ | `target` |
| $B$ | OLS 係數 | $(1 + kp, k)$ | `self.B_` |
| $A_j$ | $B$ 的第 $1 + (j-1)k$ 到 $jk$ 列**轉置** | $(k, k)$ | `self.coefs_[j − 1]` |
| $U$ | 殘差 | $(n - p, k)$ | `self.resid_` |

**下標公式**：
- TODO 3：`X = np.column_stack([np.ones(n − p)] + [Y[p − j : n − j] for j in range(1, p + 1)])`（`Y[p − j : n − j]` 是 $(n - p, k)$ 的一塊，column_stack 會把它的 $k$ 欄接上去）；`target = Y[p:]`。

#### 最小例子：TODO 3

$Y = \begin{pmatrix}1 & 10\\2 & 20\\3 & 30\\4 & 40\end{pmatrix}$（$n = 4$、$k = 2$），$p = 1$。

1. **正確答案**：$X = \begin{pmatrix}1 & 1 & 10\\1 & 2 & 20\\1 & 3 & 30\end{pmatrix}$（常數、上一期的兩條）、target $= \begin{pmatrix}2 & 20\\3 & 30\\4 & 40\end{pmatrix}$。
2. **積木**：`Y[p − j : n − j]` 在 $j = 1$ 時是什麼。
3. **留給你**：TODO 3。

In [ ]:
# 最小例子：TODO 3
Y_mini = np.array([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0], [4.0, 40.0]])
print("j = 1：Y[0:3] =\n", Y_mini[0:3])
# 寫完 TODO 3、執行過下一格之後，取消下面的註解來對照：
# print(var_design(Y_mini, 1))

### 3B. OLS、殘差共變異、AIC

- $B = \arg\min\lVert\text{target} - XB\rVert^2$：`np.linalg.lstsq(X, target)` —— $k$ 條方程式**共用同一個 $X$**，一次解完（前置複習 3）。
- $A_j = B[1 + (j-1)k : 1 + jk]^\top$（轉置：$B$ 的那一塊是「lag $j$ 的序列 $l$（列）→ 目標 $i$（欄）」，$A_j[i, l]$ 要反過來）。
- $\hat\Sigma_u = U^\top U / T$（$T = n - p$，最大概似的分母）；AIC $= \log\det\hat\Sigma_u + 2(k^2 p + k)/T$（statsmodels 的定義）。

**下標公式**：
- TODO 4：`B = np.linalg.lstsq(X, target, rcond=None)[0]`；`coefs = np.stack([B[1 + (j − 1) * k : 1 + j * k].T for j in range(1, p + 1)])`；`U = target − X @ B`；`sigma_u = U.T @ U / T`；`aic = np.log(np.linalg.det(sigma_u)) + 2 * (k * k * p + k) / T`。

#### 最小例子：TODO 4

資料由 $\mathbf y_t = \mathbf c + A\mathbf y_{t-1}$（**沒有雜訊**）產生，$\mathbf c = (1, 0)$、$A = \begin{pmatrix}0.5 & 0\\0.1 & 0.5\end{pmatrix}$、$\mathbf y_0 = (0, 5)$，10 期。OLS 應該**剛好**估回 $\mathbf c$ 與 $A$（殘差 0）。

In [ ]:
from typing import Self


def var_design(Y: np.ndarray, p: int) -> tuple[np.ndarray, np.ndarray]:
    """回傳 (X (n − p, 1 + kp)，target (n − p, k))。"""
    Y = np.asarray(Y, dtype=float)
    n = len(Y)
    # TODO 3（程式）：見 3A 的下標公式
    raise NotImplementedError("TODO 3：設計矩陣")
    return X, target


class VARScratch:
    """VAR(p)，有常數。屬性名稱仿 statsmodels：B_（= params）、intercept_、coefs_ (p, k, k)、sigma_u_、aic_、resid_。"""

    def __init__(self, p: int) -> None:
        self.p = p

    def fit(self, Y: np.ndarray) -> Self:
        Y = np.asarray(Y, dtype=float)
        p, k = self.p, Y.shape[1]
        X, target = var_design(Y, p)
        T = len(target)
        # TODO 4（程式）：B、coefs（p 個 A_j）、殘差 U、sigma_u、aic（見 3B 的下標公式）
        raise NotImplementedError("TODO 4：OLS 與 AIC")
        self.B_, self.intercept_, self.coefs_ = B, B[0], coefs
        self.resid_, self.sigma_u_, self.aic_, self.nobs_ = U, sigma_u, aic, T
        self.Y_ = Y
        return self

    def forecast(self, y_last: np.ndarray, h: int) -> np.ndarray:
        """y_last：最後 p 期 (p, k)（舊 → 新）。回傳遞迴的 h 步預測 (h, k)。"""
        hist = [np.asarray(r, dtype=float) for r in y_last]
        out = []
        for _ in range(h):
            # TODO 5（程式）：y_next = c + Σ_j A_j @ hist[−j]（j = 1..p）；append 到 hist、out
            raise NotImplementedError("TODO 5：遞迴預測")
        return np.array(out)

    def one_step(self, Y_all: np.ndarray, start: int, steps: int) -> np.ndarray:
        """滾動的一步預測：第 t = start..start + steps − 1 期，用真實的 Y_all[t − p : t]。"""
        return np.array([self.forecast(Y_all[t - self.p : t], 1)[0] for t in range(start, start + steps)])

    def irf(self, h: int) -> np.ndarray:
        """脈衝響應 Ψ_0..Ψ_h (h + 1, k, k)：Ψ_0 = I，Ψ_s = Σ_{j=1}^{min(s, p)} Ψ_{s−j} A_j。"""
        k = self.coefs_.shape[1]
        Psi = [np.eye(k)]
        for s in range(1, h + 1):
            # TODO 6（程式）：Ψ_s（見 docstring；注意矩陣乘法的順序是 Ψ_{s−j} @ A_j）
            raise NotImplementedError("TODO 6：脈衝響應")
        return np.array(Psi)

In [ ]:
# 最小例子：TODO 4、5、6
c_true, A_true = np.array([1.0, 0.0]), np.array([[0.5, 0.0], [0.1, 0.5]])
Y_sim = [np.array([0.0, 5.0])]
for _ in range(9):
    Y_sim.append(c_true + A_true @ Y_sim[-1])
Y_sim = np.array(Y_sim)
print("模擬的資料（前 4 期）：\n", Y_sim[:4].round(4))
# 寫完 TODO 4–6 之後，取消下面的註解來對照：
# v_mini = VARScratch(1).fit(Y_sim); print("c =", v_mini.intercept_.round(6), "\nA =\n", v_mini.coefs_[0].round(6), "\nmax |resid| =", np.abs(v_mini.resid_).max())   # 應為 c = (1, 0)、A = A_true、殘差 ≈ 0
# print(v_mini.forecast(np.array([[2.0, 1.0]]), 2))    # 應為 [[2. 0.7] [2. 0.55]]（前置複習 2：c + A @ (2, 1) = (2.0, 0.7)）
# print(v_mini.irf(2).round(4))                        # 應為 I、A、A² = [[0.25 0] [0.1 0.25]]

### 3E. 對照 statsmodels；實驗

statsmodels：`VAR(Y).fit(p)` 的 `params`（= $B$）、`aic`、`forecast(y_last, h)`、`irf(h).irfs`（= $\Psi$）都應該**完全一樣**。

| 實驗 | 問題 |
| --- | --- |
| 1 | 選 $p$：AIC、BIC 隨 $p$ 怎麼變？（同第 2 章：要用同樣的資料點比） |
| 2 | Granger 因果：「消費、投資的過去」對預測 GDP 有沒有幫助？每一對都檢定一次 |
| 3 | 脈衝響應：投資多 1 個百分點的衝擊，GDP、消費之後幾季怎麼變？正交化（Cholesky）之後呢？變數的順序有影響嗎？ |

In [ ]:
from statsmodels.tsa.api import VAR

Y_tr = g_train.to_numpy()
for p in [1, 2, 4]:
    mine = VARScratch(p).fit(Y_tr)
    sm = VAR(Y_tr).fit(p)
    print(f"p = {p}：✅ B 差 {np.abs(mine.B_ - sm.params).max():.1e}；✅ AIC 差 {abs(mine.aic_ - sm.aic):.1e}；"
          f"✅ 8 步預測差 {np.abs(mine.forecast(Y_tr[-p:], 8) - sm.forecast(Y_tr[-p:], 8)).max():.1e}；✅ IRF 差 {np.abs(mine.irf(10) - sm.irf(10).irfs).max():.1e}")

In [ ]:
# 實驗 1：選 p —— 所有 p 都用同樣的 T = n − P_MAX 期（每個 p 都從 Y_tr[P_MAX − p:] 開始）
P_MAX = 8
rows = []
for p in range(1, P_MAX + 1):
    mdl = VARScratch(p).fit(Y_tr[P_MAX - p :])
    T, k = mdl.nobs_, K
    n_params = k * k * p + k
    rows.append({"p": p, "AIC (common sample)": mdl.aic_, "BIC (common sample)": np.log(np.linalg.det(mdl.sigma_u_)) + np.log(T) * n_params / T,
                 "AIC (own sample)": VARScratch(p).fit(Y_tr).aic_, "# params": n_params})
sel = pd.DataFrame(rows).set_index("p")
display(sel.round(4))
sm_sel = VAR(Y_tr).select_order(P_MAX)
print("statsmodels select_order：", sm_sel.selected_orders)
P_AIC, P_BIC = int(sel["AIC (common sample)"].idxmin()), int(sel["BIC (common sample)"].idxmin())
print("手寫：AIC 選 p =", P_AIC, "；BIC 選 p =", P_BIC)

In [ ]:
# 實驗 2：Granger 因果（每一對：「source 的過去」對預測 target 有沒有幫助），用 AIC 選的 p
sm_p = VAR(g_train).fit(P_AIC)
gc_rows = []
for target, source in itertools.permutations(COLS, 2):
    res = sm_p.test_causality(target, [source], kind="f")
    gc_rows.append({"source → target": f"{source} → {target}", "F": res.test_statistic, "p-value": res.pvalue})
display(pd.DataFrame(gc_rows).set_index("source → target").round(4))

In [ ]:
# 實驗 3：脈衝響應（10 季）
var_p = VARScratch(P_AIC).fit(Y_tr)
Psi = var_p.irf(10)
P_chol = np.linalg.cholesky(var_p.sigma_u_)  # 正交化：衝擊 = P @ (獨立的單位衝擊)
Psi_orth = Psi @ P_chol
fig, axes = plt.subplots(2, K, figsize=(16, 6), sharex=True)
for l, src in enumerate(COLS):
    for i, tgt in enumerate(COLS):
        axes[0, l].plot(Psi[:, i, l], "o-", ms=3, label=tgt)
        axes[1, l].plot(Psi_orth[:, i, l], "o-", ms=3, label=tgt)
    axes[0, l].set_title(f"non-orthogonal: +1 in {src}")
    axes[1, l].set_title(f"orthogonal (Cholesky, order {COLS}): shock to {src}")
    for r in (0, 1):
        axes[r, l].axhline(0, color="k", lw=0.5)
axes[0, 0].legend()
axes[1, 0].set_xlabel("quarters after the shock")
plt.tight_layout()
plt.show()
print("殘差的相關係數：\n", np.round(var_p.sigma_u_ / np.sqrt(np.outer(np.diag(var_p.sigma_u_), np.diag(var_p.sigma_u_))), 3))
rev = list(reversed(range(K)))
P_rev = np.linalg.cholesky(var_p.sigma_u_[np.ix_(rev, rev)])
print("順序反過來（realinv, realcons, realgdp）時，投資衝擊對 GDP 的同期效果：", round(float((Psi[0][np.ix_(rev, rev)] @ P_rev)[2, 0]), 3),
      "；原本的順序：", round(float(Psi_orth[0, 0, 2]), 3))

In [ ]:
# TODO 7（問答）TODO 3–6 與對照
#
# (1) 對照 statsmodels：B、AIC、預測、IRF 的差異各是多少？
# (2) 為什麼「每條方程式各自 OLS」就等於整個 VAR 的最大概似估計（右邊的變數都一樣時）？（提示：k 條方程式的 X 相同 —— 第一部分第 2 章多元回歸）
# (3) 脈衝響應 Ψ_1 = A_1：Ψ_1[i, l] 的意義是什麼？（「第 0 季序列 l 多 1、第 1 季序列 i 多多少」）
# 答：

In [ ]:
# TODO 8（問答）實驗 1–3
#
# (1) 實驗 1：
#     (a) common sample 的 AIC、BIC 各選幾個 lag？BIC 為什麼通常選得比較少？
#     (b) 「own sample」的 AIC（每個 p 用自己的資料點）會偏好大的 p 還是小的 p？跟第 2 章的教訓一樣嗎？
# 答：
#
# (2) 實驗 2：哪幾對的 p 值 < 0.05？「投資 Granger-cause GDP」代表投資「造成」GDP 嗎？Granger 因果在說什麼？
#     （提示：只是「預測力」—— 過去的投資對預測未來的 GDP 有沒有額外的幫助；可能兩者都受第三個因素（利率、預期）影響）
# 答：
#
# (3) 實驗 3：
#     (a) 非正交化：投資 +1 的衝擊，GDP 在第 1、2 季怎麼反應？多久回到 0？為什麼一定會回到 0？（平穩）
#     (b) 殘差的同期相關有多高？正交化的意思是「把同一季一起發生的部分，歸給排在前面的變數」——
#         順序反過來時，投資衝擊對 GDP 的同期效果差多少？這代表正交化的脈衝響應「客觀」嗎？
# 答：

---
## 4. 套件實作：statsmodels `VAR` 的其他功能

| 功能 | 方法 | 用途 |
| --- | --- | --- |
| 穩定性 | `res.is_stable()`、companion matrix 的特徵值 | 所有特徵值的絕對值 < 1 → 平穩 |
| 殘差白雜訊 | `res.test_whiteness(nlags)` | 多變量的 Ljung-Box（Portmanteau） |
| 常態性 | `res.test_normality()` | Jarque-Bera |
| 預測區間 | `res.forecast_interval(y_last, h, alpha)` | |
| 預測誤差變異分解（FEVD） | `res.fevd(h)` | $h$ 季後的預測誤差，有多少比例來自各個（正交化的）衝擊 |

In [ ]:
res = VAR(g_train).fit(P_AIC)
companion_eig = np.abs(np.linalg.eigvals(res.coefs[0] if P_AIC == 1 else np.vstack([np.hstack(list(res.coefs)), np.eye(K * (P_AIC - 1), K * P_AIC)])))
print("穩定？", res.is_stable(), "；companion matrix 特徵值的最大絕對值：", round(float(companion_eig.max()), 3))
print(res.test_whiteness(nlags=8).summary())
print(res.test_normality().summary())
fevd = res.fevd(8)
print("\n8 季後的預測誤差變異分解（列：被預測的序列；欄：衝擊來源）：")
display(pd.DataFrame(fevd.decomp[:, -1, :], index=COLS, columns=[f"shock: {c}" for c in COLS]).round(3))

mid, lower, upper = res.forecast_interval(Y_tr[-P_AIC:], steps=TEST_Q, alpha=0.05)
fig, axes = plt.subplots(1, K, figsize=(17, 3.5))
for i, c in enumerate(COLS):
    axes[i].plot(growth.index[-40:].to_timestamp(), growth[c].iloc[-40:], color="k", label="actual")
    t_idx = g_test.index.to_timestamp()
    axes[i].plot(t_idx, mid[:, i], color="tab:red", label="VAR multi-step")
    axes[i].fill_between(t_idx, lower[:, i], upper[:, i], color="tab:red", alpha=0.2)
    axes[i].set_title(c)
axes[0].legend(fontsize=8)
plt.show()

In [ ]:
# TODO 9（問答）
#
# (1) 模型穩定嗎？殘差的白雜訊、常態性檢定各是什麼結論？如果常態性被拒絕，對哪些東西有影響（點預測？預測區間？）
# (2) FEVD：8 季後 GDP 的預測誤差，有多少來自自己的衝擊、多少來自消費、投資？（注意：這是 Cholesky 正交化的結果，跟順序有關）
# (3) 多步預測（從 train 的最後一季往後 12 季）很快就變成一條平線 —— 平在什麼值？為什麼？它抓得到 2008 年的衰退嗎？
# 答：

---
## 5. 調參：滾動起點的一步預測 CV

候選：naive、單變數 AR($p$)（$p = 1, 2$）、VAR($p$)（$p = 1, 2, 4$）。在 **train 的最後 40 季**做滾動一步預測：每 4 季重新估計一次參數（只用起點之前的資料），預測接下來 4 季。全程不碰 test。

In [ ]:
def cv_one_step(make_model, Y: np.ndarray, first: int, step: int = 4) -> np.ndarray:
    """回傳 (len(Y) − first, K) 的一步預測；每 step 期用 Y[:origin] 重新估計。make_model(Y_fit) → 一個有 predict_next(Y_hist) 的物件。"""
    preds = []
    for origin in range(first, len(Y), step):
        mdl = make_model(Y[:origin])
        for t in range(origin, min(origin + step, len(Y))):
            preds.append(mdl(Y[:t]))
    return np.array(preds)


def var_maker(p: int):
    def make(Y_fit: np.ndarray):
        v = VARScratch(p).fit(Y_fit)
        return lambda Y_hist: v.forecast(Y_hist[-p:], 1)[0]
    return make


def uar_maker(p: int):
    def make(Y_fit: np.ndarray):
        betas = []
        for i in range(K):
            z = Y_fit[:, i]
            X = np.column_stack([np.ones(len(z) - p)] + [z[p - j : len(z) - j] for j in range(1, p + 1)])
            betas.append(np.linalg.lstsq(X, z[p:], rcond=None)[0])
        return lambda Y_hist: np.array([b[0] + sum(b[j] * Y_hist[-j, i] for j in range(1, p + 1)) for i, b in enumerate(betas)])
    return make


FIRST = len(Y_tr) - 40
cands = {"naive": lambda Y_fit: (lambda Y_hist: Y_hist[-1]), "univariate AR(1)": uar_maker(1), "univariate AR(2)": uar_maker(2),
         "VAR(1)": var_maker(1), "VAR(2)": var_maker(2), "VAR(4)": var_maker(4)}
cv_rows = {}
for name, maker in cands.items():
    pred = cv_one_step(maker, Y_tr, FIRST)
    actual = Y_tr[FIRST:]
    err = actual - pred
    cv_rows[name] = {**{f"{c} rmse": float(np.sqrt((err[:, i] ** 2).mean())) for i, c in enumerate(COLS)}, "mean rmse (scaled)": float(np.mean(np.sqrt((err**2).mean(axis=0)) / Y_tr.std(axis=0)))}
cv_table = pd.DataFrame(cv_rows).T.sort_values("mean rmse (scaled)")
display(cv_table.round(4))
chosen = cv_table.index[0]
print("CV 最好的模型：", chosen)

In [ ]:
# TODO 10（問答）
#
# (1) CV 的排名：VAR 有贏過單變數 AR 嗎？哪一條序列的 RMSE 差最多？跟實驗 2（Granger 因果）的結果一致嗎？
# (2) VAR(4) 的參數比 VAR(1) 多幾倍？它在 CV 上比較好還是比較差？為什麼參數多的 VAR 容易過擬合？（k² p 個係數）
# (3) 「mean rmse (scaled)」把每條的 RMSE 除以它的標準差再平均 —— 為什麼不能直接把三條的 RMSE 平均？
# 答：

---
## 6. 測試集評估

用第 5 節選出的模型（在完整的 train 上估計一次，參數固定），在 test 的 12 季做滾動的一步預測，只評估一次。成績寫進 `results/macro_growth.csv`。

In [ ]:
from learning_map.results import compare, log_result

BENCH = "macro_growth"
final_pred = {}
for name in ["naive", "univariate AR(1)", "univariate AR(2)", "VAR(1)", "VAR(2)", chosen]:
    model_fn = cands[name](Y_tr)
    final_pred[name] = np.array([model_fn(Y_all[:N_TR + s]) for s in range(TEST_Q)])
for name, pred in final_pred.items():
    label = f"{name} (CV-selected)" if name == chosen else name
    test_table.loc[label] = score_one_step(pred, name)
    log_result(BENCH, label, test_table.loc[label].to_dict(), horizon=1, test=f"{g_test.index[0]}-{g_test.index[-1]}")
display(test_table)
compare(BENCH, "mean mase", higher_is_better=False)

---
## 7. 診斷

1. test 期間每一季的一步預測 vs 實際（2008 年金融危機）。
2. VAR 的殘差：每條序列的 ACF、殘差之間的同期相關。

In [ ]:
best_pred = final_pred[chosen]
fig, axes = plt.subplots(2, K, figsize=(17, 6.5))
t_idx = g_test.index.to_timestamp()
for i, c in enumerate(COLS):
    axes[0, i].plot(t_idx, g_test[c].to_numpy(), "o-", color="k", label="actual")
    axes[0, i].plot(t_idx, best_pred[:, i], "o--", color="tab:red", label=chosen)
    axes[0, i].plot(t_idx, final_pred["naive"][:, i], "x:", color="gray", label="naive")
    axes[0, i].set_title(f"{c}: one-step forecasts in test")
    resid_i = var_p.resid_[:, i]
    r = np.array([1.0] + [np.corrcoef(resid_i[L:], resid_i[:-L])[0, 1] for L in range(1, 13)])
    axes[1, i].bar(np.arange(13), r)
    axes[1, i].axhline(2 / np.sqrt(len(resid_i)), color="gray", ls=":")
    axes[1, i].axhline(-2 / np.sqrt(len(resid_i)), color="gray", ls=":")
    axes[1, i].set_title(f"VAR({P_AIC}) residual ACF: {c}")
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()
err_tbl = pd.DataFrame({c: g_test[c].to_numpy() - best_pred[:, i] for i, c in enumerate(COLS)}, index=g_test.index.astype(str))
display(err_tbl.round(2).T)

In [ ]:
# TODO 11（問答，本章總結）
#
# (1) 測試集：選出的模型的平均 MASE 是多少？比 naive、單變數 AR 好嗎？哪一條序列進步最多？
# (2) 誤差表：誤差最大的是哪幾季？是預測太樂觀還是太悲觀？為什麼任何「用過去的線性關係」的模型都很難預測 2008Q4？
# (3) VAR 的限制：
#     (a) k 條序列、p 個 lag → k² p 個係數。如果 k = 20（真實的總經模型），p = 4，有多少個係數？季資料 50 年只有 200 季 —— 會發生什麼事？
#     (b) 這一章用成長率（差分過）。如果 GDP 與消費的「水準」長期一起移動（共整合），只用成長率會丟掉什麼資訊？（提示：VECM）
# (4) 第 1～3 章的比較：指數平滑、ARIMA、VAR 各適合什麼樣的問題？
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

美國總經季成長率（GDP、消費、投資；train 1959Q2–2006Q3、test 2006Q4–2009Q3，滾動一步預測）

| 方法 | 做法 | 參數個數 | CV mean RMSE (scaled) | test mean MASE |
| --- | --- | --- | --- | --- |
| naive | | 0 | | |
| 單變數 AR(1)／AR(2) | | | | |
| VAR(1) | 每條方程式 OLS | | | |
| VAR(2)／VAR(4) | | | | |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ VAR 的平穩條件與 companion form；VAR(p) = VAR(1) 的堆疊形式
- ☐ 結構 VAR（SVAR）：用經濟理論而不是 Cholesky 順序來識別衝擊
- ☐ 共整合與 VECM（Engle & Granger 1987；Johansen 檢定）
- ☐ 脈衝響應的信賴區間（漸近、bootstrap）

**延伸**
- ☐ 貝氏 VAR（Minnesota prior）：大量序列時的收縮
- ☐ 因子模型（dynamic factor model）：幾百條總經序列
- ☐ 高頻資料混頻（MIDAS）、即時預測（nowcasting）